# Preprocessing Technique: Handling Missing Data & Duplicate Records
### IT2011 — Progress Review I: Data Preprocessing and EDA
**Presented by:** Member 1 — *[Full Name, IT Number]*
**Assigned dataset:** Rotten Tomatoes Movie Review Dataset (Cornell)

This notebook covers my individually-owned preprocessing technique for our group's project,
as required for Progress Review I: technique explanation, justification, implementation, and
an interpreted EDA visualization.


## Shared Setup

This cell is identical across every member's notebook so each person's notebook can run
independently. It loads the assigned dataset and converts it to a pandas DataFrame.


In [ ]:
!pip install -q datasets scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_style("whitegrid")

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)
train_df.head()


## 1. Technique Explanation

**Handling missing data** means identifying and resolving gaps or invalid entries in a dataset
before it is used for modeling. For a text-classification dataset like ours, "missing data"
includes: null/NaN values in the `text` or `label` columns, empty or whitespace-only review
strings, and exact duplicate rows (which are a related data-quality issue — they can silently
bias a model toward whichever review happens to be repeated).

## 2. Justification for This Dataset

Our dataset (Rotten Tomatoes Movie Review Dataset) is a curated academic benchmark, so it is
reasonable to *expect* it to be clean — but that expectation must be verified, not assumed.
If left unchecked, missing or duplicate rows would: (a) cause errors when vectorizing text
that is `NaN`, or (b) quietly inflate the apparent size and balance of the training set if a
review is duplicated. Verifying this is a required, reportable step regardless of the result.


## 3. Implementation

In [ ]:
# Check for missing values in all three splits
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"--- {name} ---")
    print(df.isna().sum())
    print()


In [ ]:
# Check for duplicate rows and empty/whitespace-only review text
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    n_duplicates = df.duplicated(subset=["text"]).sum()
    n_empty = df["text"].str.strip().eq("").sum()
    print(f"{name:12s} | duplicate texts: {n_duplicates:4d} | empty reviews: {n_empty:4d}")


## 4. EDA Visualization & Interpretation

The bar chart below summarizes the data-quality check results across all three splits.


In [ ]:
issue_counts = pd.DataFrame({
    "Split": ["train", "validation", "test"],
    "Missing values": [train_df.isna().sum().sum(), val_df.isna().sum().sum(), test_df.isna().sum().sum()],
    "Duplicate texts": [train_df.duplicated(subset=["text"]).sum(),
                         val_df.duplicated(subset=["text"]).sum(),
                         test_df.duplicated(subset=["text"]).sum()],
})
issue_counts_melted = issue_counts.melt(id_vars="Split", var_name="Issue", value_name="Count")

plt.figure(figsize=(7, 4))
sns.barplot(data=issue_counts_melted, x="Split", y="Count", hue="Issue")
plt.title("Data Quality Check: Missing Values & Duplicates by Split")
plt.ylim(0, max(1, issue_counts_melted["Count"].max() + 1))
plt.show()

issue_counts


**Interpretation:** [Fill in after running — expected result: all bars at zero, confirming
the dataset has no missing values or duplicate reviews across any split. This means no
imputation or deduplication step is required before modeling, though this check should still
be shown in the viva as evidence of due diligence rather than an assumption.]
